In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
from brain_image.data import EEGDataset, EEGDatasetConfig


ds = EEGDataset(EEGDatasetConfig(subs=[8]), "test")
dst = EEGDataset(EEGDatasetConfig(subs=[8]), "train")
ds[1]

{'img_path': 'data/things-eeg2/imgs/test_images/00002_antelope/antelope_01b.jpg',
 'eeg_data': tensor([[-0.0442, -0.1725, -0.0251,  ...,  0.3456,  0.4039,  0.4341],
         [ 0.0552,  0.0256,  0.0539,  ...,  0.4109,  0.3808,  0.3658],
         [-0.1556, -0.0982,  0.0302,  ...,  0.0292, -0.0312,  0.0992],
         ...,
         [ 0.1606,  0.2817,  0.3878,  ...,  0.0116, -0.0630,  0.0147],
         [ 0.0894,  0.1868,  0.3480,  ...,  0.0765,  0.0410, -0.0190],
         [-0.1024, -0.0676, -0.0699,  ..., -0.1189, -0.1080, -0.1614]]),
 'idx': tensor(1),
 'sub': 8}

In [16]:
import torch
group_samples = torch.load("tmp/group_samples.pt")
group_samples_train = torch.load("tmp/group_samples_train.pt")
print(group_samples[0].keys())
s = group_samples[15]
s["eeg"]

dict_keys(['eeg', 'label', 'text', 'text_features', 'img', 'img_features', 'idx'])


tensor([[ 0.1369, -0.0011,  0.0260,  ...,  0.5450,  0.5119,  0.5970],
        [ 0.0023,  0.0426,  0.0756,  ...,  0.5368,  0.4932,  0.4050],
        [-0.0150, -0.0204,  0.0049,  ...,  0.0762,  0.1212,  0.0206],
        ...,
        [ 0.1710,  0.2278,  0.3594,  ..., -0.1812, -0.1349, -0.0932],
        [ 0.1145,  0.1762,  0.2550,  ...,  0.0202, -0.1166, -0.1716],
        [ 0.0301,  0.0130,  0.0053,  ..., -0.2021, -0.2140, -0.2416]])

In [8]:
ds[15]["eeg_data"]

tensor([[ 0.1369, -0.0011,  0.0260,  ...,  0.5450,  0.5119,  0.5970],
        [ 0.0023,  0.0426,  0.0756,  ...,  0.5368,  0.4932,  0.4050],
        [-0.0150, -0.0204,  0.0049,  ...,  0.0762,  0.1212,  0.0206],
        ...,
        [ 0.1710,  0.2278,  0.3594,  ..., -0.1812, -0.1349, -0.0932],
        [ 0.1145,  0.1762,  0.2550,  ...,  0.0202, -0.1166, -0.1716],
        [ 0.0301,  0.0130,  0.0053,  ..., -0.2021, -0.2140, -0.2416]])

In [18]:
torch.allclose(group_samples_train[131]["eeg"], dst[131]["eeg_data"])

True

In [19]:
group_samples_train[131]["img"], dst[131]["img_path"]

('../data/things-eeg2/images_set/training_images/00004_acorn/acorn_03s.jpg',
 'data/things-eeg2/imgs/training_images/00004_acorn/acorn_03s.jpg')

In [3]:
from brain_image.data import load_eeg_data
from pathlib import Path
raw_eeg, *_ = load_eeg_data(Path("data/things-eeg2/eeg/sub-08/preprocessed_eeg_training.npy"))

In [11]:
raw_eeg_test, *_ =  load_eeg_data(Path("data/things-eeg2/eeg/sub-08/preprocessed_eeg_test.npy"))

In [12]:
raw_eeg.shape, raw_eeg_test.shape

(torch.Size([16540, 4, 63, 250]), torch.Size([200, 80, 63, 250]))

In [13]:
print(raw_eeg.mean(), raw_eeg.std())

tensor(-0.0190) tensor(1.0974)


In [4]:
from brain_image.data import preprocess_eeg_data

eeg = preprocess_eeg_data(raw_eeg)

In [5]:
com_idx = 4000
v1 = ds[com_idx]["eeg_data"]
v2 = eeg[com_idx]
import torch
torch.allclose(v1, v2)


False

In [6]:
ds[7]["img_path"]

'data/things-eeg2/imgs/training_images/00001_aardvark/aardvark_08s.jpg'

In [7]:
eeg.mean(), eeg.std()

(tensor(2.4742e-10), tensor(0.7071))

In [8]:
import numpy as np
eeg2 = (eeg - eeg.mean(dim=0, keepdim=True)) * (1 / (np.sqrt(2) * eeg.std(dim=0, keepdim=True)))
eeg2.mean(), eeg2.std()
eeg2[:10, :10]

tensor([[[ 0.4194,  0.5192,  0.0936,  ...,  0.8014,  1.6233,  1.8319],
         [-0.2249,  0.4113,  0.6163,  ...,  1.5677,  1.3518,  1.6217],
         [-0.4522,  0.0561,  0.2565,  ...,  0.4089, -0.5389, -0.5065],
         ...,
         [ 0.6043,  0.1838, -0.3828,  ..., -0.6086,  1.0633,  1.0087],
         [-1.4992, -0.9889, -0.6074,  ...,  0.9058, -0.0229,  0.5805],
         [-0.7916, -0.7862,  0.4455,  ...,  0.6124,  0.2314,  0.4069]],

        [[ 0.4941,  0.9092, -1.2277,  ..., -0.9896,  0.6131,  0.3494],
         [-0.0206, -1.2433,  0.1391,  ...,  0.7254,  0.6237, -0.1362],
         [-0.7758, -0.5140, -0.3268,  ...,  0.2754, -0.4520, -0.0416],
         ...,
         [-0.2991, -0.6237,  0.0971,  ..., -0.1169,  0.4488,  0.1036],
         [-1.1311,  0.4973,  1.1889,  ...,  1.4277,  0.4803, -0.5132],
         [-0.4090,  0.4383,  1.0779,  ...,  1.0466, -0.0354, -0.3108]],

        [[-0.3275,  0.3220,  0.4334,  ...,  0.0351,  0.5040,  0.6450],
         [ 0.2368, -0.1928,  1.3649,  ...,  0